<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/07_optimizations_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Capstone Optimizations (package v0.2)

`06` established the honest baseline: a modestly positive congressional signal, a
gov-contracts component that ran backwards, and a composite whose market-neutral alpha is
thin. This notebook acts on that, upgrading the package to `v0.2` and testing four
optimizations end to end.

1. **Score-ranked priced universe.** The earlier backtests priced an alphabetical slice of
   the union. Here the universe is the top names *by composite score*, so the IC is measured
   on the ranking that actually matters.
2. **Long-only conviction curve and a beta/alpha split.** A long-only top-quintile portfolio
   is what the vendor trackers report. Decomposing its return into market beta and residual
   alpha shows where a headline CAGR really comes from.
3. **Static high-conviction congress.** A feature tilt toward size, committee alignment, and
   cluster — the attributes most associated with informed buying.
4. **Dynamic member-skill weighting.** An empirical-Bayes estimate of each member's realized
   edge, re-fit walk-forward so a member earns influence only as their record matures.

The package changes are backward-compatible — the new behavior sits behind defaults — so
`01`–`06` keep working unchanged.


## Setup

In [2]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests yfinance scipy

import subprocess, os, sys, logging
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data + real prices

Working in: /content/quant-equity-research


## Upgrade the package to v0.2

This is the one place the `v0.2` code is written and committed. `engine.py` gains the
`beta_alpha` split, `congress.py` gains the high-conviction tilt and an optional per-member
weighting, `member_skill.py` is new, and `client.py` is hardened so a rate-limited or non-tabular
live response fails with a clear message and a bulk-to-recent fallback rather than an opaque
pandas error. After this commit the repo's `src/` files are the
single source of truth — the research notebooks only clone and import.


In [3]:
open("src/quant_research/backtest/engine.py", "w").write('"""Event-study backtester for a cross-sectional signal.\n\nOn each rebalance date the engine scores the universe, sorts names into buckets by\nscore, and measures their forward return over a fixed horizon. Aggregated across\ndates, this answers the core question: do higher-scoring names earn higher forward\nreturns? It reports three complementary views:\n\n  - bucket means: average forward return by score bucket (monotonic is the goal)\n  - information coefficient (IC): rank correlation of score with forward return,\n    averaged across dates, with the share of dates where it is positive\n  - a long-top-bucket equity curve with standard risk metrics\n\nThe engine takes a `score_fn(date) -> Series[ticker -> score]`, so it is agnostic to\nwhich signal it is testing.\n"""\nimport numpy as np\nimport pandas as pd\n\nfrom .prices import forward_returns\n\n\ndef _spearman(a, b):\n    if a.nunique() < 2 or b.nunique() < 2:\n        return np.nan\n    return a.rank().corr(b.rank())\n\n\ndef event_study(score_fn, prices, rebalance_dates, horizon=21, n_buckets=5):\n    bucket_rows, ic_rows, ls_rows = [], [], []\n    for d in rebalance_dates:\n        scores = score_fn(d)\n        if scores is None or len(scores) < n_buckets:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = pd.concat([scores.rename("score"), fwd.rename("fwd")], axis=1).dropna()\n        if len(df) < n_buckets:\n            continue\n        df["bucket"] = pd.qcut(df["score"].rank(method="first"), n_buckets, labels=False)\n        means = df.groupby("bucket")["fwd"].mean()\n        for b, m in means.items():\n            bucket_rows.append({"date": d, "bucket": int(b), "fwd": m})\n        ls_rows.append({"date": d, "long_short": means.get(n_buckets - 1) - means.get(0)})\n        ic_rows.append({"date": d, "ic": _spearman(df["score"], df["fwd"])})\n\n    buckets = pd.DataFrame(bucket_rows)\n    ls = pd.DataFrame(ls_rows)\n    ic = pd.DataFrame(ic_rows).dropna()\n\n    summary = {\n        "n_periods": len(ls),\n        "mean_ic": ic["ic"].mean() if len(ic) else np.nan,\n        "ic_positive_share": (ic["ic"] > 0).mean() if len(ic) else np.nan,\n        "mean_long_short": ls["long_short"].mean() if len(ls) else np.nan,\n        "ls_positive_share": (ls["long_short"] > 0).mean() if len(ls) else np.nan,\n    }\n    bucket_means = (buckets.groupby("bucket")["fwd"].mean()\n                    if len(buckets) else pd.Series(dtype=float))\n    return {"summary": summary, "bucket_means": bucket_means,\n            "long_short": ls, "ic": ic}\n\n\ndef feature_buckets(score_fn_full, prices, rebalance_dates, feature, horizon=21):\n    """Forward return split by whether a 0/1 feature is present (e.g. committee_alignment>0)."""\n    on, off = [], []\n    for d in rebalance_dates:\n        panel = score_fn_full(d)              # full feature DataFrame indexed by ticker\n        if panel is None or panel.empty:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = panel.join(fwd.rename("fwd")).dropna(subset=["fwd"])\n        if df.empty:\n            continue\n        flag = df[feature] > 0\n        on.extend(df.loc[flag, "fwd"].tolist())\n        off.extend(df.loc[~flag, "fwd"].tolist())\n    return {"with_feature_mean": np.mean(on) if on else np.nan, "with_n": len(on),\n            "without_feature_mean": np.mean(off) if off else np.nan, "without_n": len(off)}\n\n\ndef long_top_curve(score_fn, prices, rebalance_dates, horizon=21, top_frac=0.2):\n    """Equity curve of equal-weighting the top score fraction on a non-overlapping\n    schedule. Entries are kept at least `horizon` trading days apart, measured against\n    the price index, so the cadence is correct whatever the rebalance-date frequency."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:          # keep holding periods from overlapping\n            continue\n        scores = score_fn(d)\n        if scores is None or len(scores) == 0:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = pd.concat([scores.rename("score"), fwd.rename("fwd")], axis=1).dropna()\n        if df.empty:\n            continue\n        k = max(1, int(round(len(df) * top_frac)))\n        rets.append(df.nlargest(k, "score")["fwd"].mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    equity = (1 + rets).cumprod()\n    return equity, rets\n\n\ndef long_short_curve(score_fn, prices, rebalance_dates, horizon=21, top_frac=0.2):\n    """Market-neutral equity curve: long the top score fraction, short the bottom,\n    on a non-overlapping schedule. Isolates the signal from market beta."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:\n            continue\n        scores = score_fn(d)\n        if scores is None or len(scores) == 0:\n            continue\n        df = pd.concat([scores.rename("score"),\n                        forward_returns(prices, d, horizon).rename("fwd")], axis=1).dropna()\n        if len(df) < 2:\n            continue\n        k = max(1, int(round(len(df) * top_frac)))\n        rets.append(df.nlargest(k, "score")["fwd"].mean() - df.nsmallest(k, "score")["fwd"].mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    return (1 + rets).cumprod(), rets\n\n\ndef benchmark_curve(prices, rebalance_dates, horizon=21):\n    """Equal-weight all available names each non-overlapping period: a universe\n    buy-and-hold proxy for separating signal alpha from market beta."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:\n            continue\n        fwd = forward_returns(prices, d, horizon).dropna()\n        if fwd.empty:\n            continue\n        rets.append(fwd.mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    return (1 + rets).cumprod(), rets\n\n\ndef metrics(rets, horizon=21):\n    """Annualized risk/return metrics from a series of per-period returns."""\n    if len(rets) == 0:\n        return {"cagr": float("nan"), "sharpe": float("nan"), "sortino": float("nan"),\n                "max_drawdown": float("nan"), "hit_rate": float("nan"), "n_trades": 0}\n    per_year = 252 / horizon\n    growth = (1 + rets).prod()\n    cagr = growth ** (per_year / len(rets)) - 1\n    vol = rets.std(ddof=1)\n    sharpe = (rets.mean() / vol * np.sqrt(per_year)) if vol > 0 else np.nan\n    downside = rets[rets < 0].std(ddof=1)\n    sortino = (rets.mean() / downside * np.sqrt(per_year)) if downside and downside > 0 else np.nan\n    equity = (1 + rets).cumprod()\n    max_dd = (equity / equity.cummax() - 1).min()\n    return {"cagr": cagr, "sharpe": sharpe, "sortino": sortino,\n            "max_drawdown": max_dd, "hit_rate": (rets > 0).mean(), "n_trades": len(rets)}\n\n\ndef beta_alpha(strategy_rets, benchmark_rets, horizon=21):\n    """Split a strategy\'s per-period returns into market beta and alpha via OLS.\n\n    Regresses strategy returns on the benchmark, r_strat = alpha + beta * r_bench + e,\n    so the long-only conviction return can be separated into the part explained by\n    market exposure (beta) and the residual selection edge (alpha). The two return\n    series are taken from the same non-overlapping schedule and aligned by position.\n    """\n    s = pd.Series(list(strategy_rets), dtype="float64").reset_index(drop=True)\n    b = pd.Series(list(benchmark_rets), dtype="float64").reset_index(drop=True)\n    n = min(len(s), len(b))\n    s, b = s.iloc[:n].to_numpy(), b.iloc[:n].to_numpy()\n    mask = ~(np.isnan(s) | np.isnan(b))\n    s, b = s[mask], b[mask]\n    if len(s) < 3 or np.std(b) == 0:\n        return {"alpha": float("nan"), "alpha_annual": float("nan"),\n                "beta": float("nan"), "r_squared": float("nan"), "n": int(len(s))}\n    beta, alpha = np.polyfit(b, s, 1)\n    pred = alpha + beta * b\n    ss_res = float(np.sum((s - pred) ** 2))\n    ss_tot = float(np.sum((s - s.mean()) ** 2))\n    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")\n    per_year = 252 / horizon\n    alpha_annual = (1 + alpha) ** per_year - 1\n    return {"alpha": float(alpha), "alpha_annual": float(alpha_annual),\n            "beta": float(beta), "r_squared": float(r2), "n": int(len(s))}\n')
open("src/quant_research/signals/congress.py", "w").write('"""Congressional cluster-buy signal.\n\nThe score rewards securities that several members of Congress are buying at once,\nwith each buy weighted by how informed it is likely to be. Features encode testable\nhypotheses; the backtest layer calibrates their weights.\n\nFeatures (each normalized across the cross-section, then weighted):\n  cluster              - distinct members buying the same name\n  size_vs_networth     - trade size relative to the member\'s net worth (conviction\n                         relative to means, controlling for the wealth confound)\n  committee_alignment  - buys by members whose committee oversees the ticker\'s sector\n  recency              - disclosures weighted toward the present\n  bipartisan           - a bonus when both parties are buying the same name\n\nTwo v0.2 levers, both off by default so earlier notebooks are unaffected:\n  high_conviction      - tilt the feature blend toward size, committee alignment, and\n                         cluster, the features most associated with informed buying\n  member_weights       - an optional per-member multiplier passed to ``compute`` that\n                         scales each member\'s contribution, used by the dynamic\n                         member-skill weighting in ``member_skill.py``\n\nMember net worth and committee membership come from the enrichment layer, which\nresolves a real implementation on live data and a synthetic one in mock mode.\n\nSignals are keyed on the DISCLOSURE (report) date, never the trade date, because a\ntrade is only actionable once it is public — which is what keeps the downstream\nbacktest free of look-ahead.\n"""\nfrom datetime import date\nimport numpy as np\nimport pandas as pd\n\nfrom .base import BaseSignal\nfrom ..enrichment.mock import MockEnrichment\nfrom ..enrichment.live import LiveEnrichment\n\nDEFAULT_WEIGHTS = {\n    "cluster": 0.30,\n    "size_vs_networth": 0.25,\n    "committee_alignment": 0.25,\n    "recency": 0.15,\n    "bipartisan": 0.05,\n}\n\n# tilt toward the features most associated with informed buying\nHIGH_CONVICTION_WEIGHTS = {\n    "cluster": 0.25,\n    "size_vs_networth": 0.35,\n    "committee_alignment": 0.30,\n    "recency": 0.10,\n    "bipartisan": 0.00,\n}\n\n\nclass CongressSignal(BaseSignal):\n    name = "congress"\n    description = ("Clustered congressional purchases, weighted by conviction "\n                   "relative to net worth and by committee-sector alignment.")\n\n    def __init__(self, client, lookback_days=30, weights=None, enrichment=None,\n                 high_conviction=False):\n        self.client = client\n        self.lookback_days = lookback_days\n        self.high_conviction = high_conviction\n        default = HIGH_CONVICTION_WEIGHTS if high_conviction else DEFAULT_WEIGHTS\n        self.weights = weights or dict(default)\n        self.enrichment = enrichment or (\n            MockEnrichment() if getattr(client, "mock", False) else LiveEnrichment())\n\n    def compute(self, as_of=None, trades=None, member_weights=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        start = as_of - pd.Timedelta(days=self.lookback_days)\n\n        trades = self.client.congress_trades() if trades is None else trades\n        buys = trades[(trades.transaction_type == "Purchase")\n                      & (trades.report_date > start)\n                      & (trades.report_date <= as_of)].copy()\n        if buys.empty:\n            return pd.DataFrame(columns=["score"])\n\n        buys["mid_amount"] = (buys.amount_min + buys.amount_max) / 2.0\n\n        # per-member multiplier (1.0 when no weights supplied -> original behavior)\n        if member_weights is not None:\n            mw = buys.representative.map(member_weights).astype(float).fillna(1.0)\n        else:\n            mw = pd.Series(1.0, index=buys.index)\n        buys["mw"] = mw.clip(lower=0.0)\n\n        nw = buys.representative.map(self.enrichment.net_worth)\n        if nw.notna().any():\n            buys["conviction"] = buys.mid_amount / nw.fillna(nw.dropna().median())\n        else:\n            buys["conviction"] = buys.mid_amount\n        buys["conviction"] *= buys["mw"]\n\n        buys["days_ago"] = (as_of - buys.report_date).dt.days\n        buys["recency_w"] = np.select(\n            [buys.days_ago <= 7, buys.days_ago <= 14], [1.0, 0.6], default=0.3) * buys["mw"]\n        aligned = np.array([int(self.enrichment.is_aligned(r, t))\n                            for r, t in zip(buys.representative, buys.ticker)], dtype=float)\n        buys["aligned_w"] = aligned * buys["mw"]\n\n        g = buys.groupby("ticker")\n        # weighted distinct-member cluster: each member counted once, scaled by skill\n        dedup = buys.drop_duplicates(["ticker", "representative"])\n        cluster = dedup.groupby("ticker").mw.sum()\n        feat = pd.DataFrame({\n            "n_buys": g.mw.sum(),\n            "cluster": cluster,\n            "size_vs_networth": g.conviction.sum(),\n            "committee_alignment": g.aligned_w.sum(),\n            "recency": g.recency_w.sum(),\n            "bipartisan": g.party.apply(lambda p: int({"D", "R"}.issubset(set(p)))),\n        })\n\n        norm = {}\n        for f in ["cluster", "size_vs_networth", "committee_alignment", "recency"]:\n            col = feat[f].astype(float)\n            spread = col.max() - col.min()\n            norm[f] = (col - col.min()) / spread if spread > 0 else col * 0.0\n        norm["bipartisan"] = feat["bipartisan"].astype(float)\n        norm = pd.DataFrame(norm)\n\n        raw = sum(self.weights[f] * norm[f] for f in self.weights)\n        feat["score"] = self._scale_0_100(raw).round(1)\n        for f in norm.columns:\n            feat[f + "_n"] = norm[f].round(3)\n        return feat.sort_values("score", ascending=False)\n')
open("src/quant_research/signals/member_skill.py", "w").write('"""Dynamic member-skill weighting for the congressional signal.\n\nA static high-conviction tilt asks *which trades look informed by their attributes*.\nThis module asks a complementary, data-driven question: *which members have actually\npicked winners*, and it lets that estimate evolve as new disclosures mature.\n\nThe estimate is built to avoid the two traps that flatter naive member rankings:\n\n  look-ahead  - a member\'s skill at a rebalance date uses only buys whose holding\n                window has already closed on or before that date, so nothing from the\n                future leaks in. Wiring this through ``event_study`` gives an honest\n                walk-forward evaluation.\n  small samples - most members disclose few trades, so a couple of lucky picks would\n                dominate a raw average. Each member\'s edge is shrunk toward zero with\n                an empirical-Bayes weight n / (n + k), so a member earns influence only\n                as their track record accumulates.\n\nA member\'s raw edge is the average *excess* forward return of their buys, where excess\nis measured against the mean forward return of all congressional buys disclosed the\nsame month. That removes the market move and isolates selection.\n"""\nimport numpy as np\nimport pandas as pd\n\n\ndef precompute_buy_returns(trades, prices, horizon=21):\n    """One-pass forward returns for every congressional buy that can be priced.\n\n    Returns a frame with the member, ticker, disclosure date, the date the holding\n    window closes (``exit_date``), the realized forward return, and its month-demeaned\n    excess. Computed once and reused across rebalance dates.\n    """\n    buys = trades[trades.transaction_type == "Purchase"].copy()\n    buys = buys[buys.ticker.isin(prices.columns)]\n    if buys.empty:\n        return pd.DataFrame(columns=["representative", "ticker", "report_date",\n                                     "exit_date", "fwd_ret", "period", "excess"])\n    idx = prices.index\n    recs = []\n    for tkr, grp in buys.groupby("ticker"):\n        px = prices[tkr].to_numpy()\n        pos = idx.searchsorted(pd.to_datetime(grp.report_date.values))\n        for rep, rd, entry in zip(grp.representative, grp.report_date, pos):\n            exit_ = entry + horizon\n            if entry < len(px) and exit_ < len(px):\n                pe, px_e = px[entry], px[exit_]\n                if pe and pe > 0 and not np.isnan(pe) and not np.isnan(px_e):\n                    recs.append((rep, tkr, pd.Timestamp(rd), idx[exit_], px_e / pe - 1.0))\n    out = pd.DataFrame(recs, columns=["representative", "ticker", "report_date",\n                                      "exit_date", "fwd_ret"])\n    if not out.empty:\n        out["period"] = out.report_date.dt.to_period("M")\n        out["excess"] = out.fwd_ret - out.groupby("period").fwd_ret.transform("mean")\n    return out\n\n\ndef member_skill(buy_rets, as_of, prior_strength=20.0):\n    """Empirical-Bayes member multipliers using only buys matured by ``as_of``.\n\n    Returns a Series mapping member -> multiplier (centered near 1.0). Members with no\n    matured history are absent, and the signal treats them as neutral.\n    """\n    if buy_rets is None or len(buy_rets) == 0:\n        return pd.Series(dtype=float)\n    matured = buy_rets[buy_rets.exit_date <= pd.Timestamp(as_of)]\n    if matured.empty:\n        return pd.Series(dtype=float)\n    grp = matured.groupby("representative").excess\n    n, raw = grp.count(), grp.mean()\n    shrunk = (n * raw) / (n + prior_strength)        # shrink toward zero edge\n    sd = shrunk.std(ddof=0)\n    z = (shrunk - shrunk.mean()) / sd if sd and sd > 0 else shrunk * 0.0\n    return np.exp(0.5 * z.clip(-2, 2))               # smooth, positive, ~[0.37, 2.72]\n\n\ndef dynamic_congress_score_fn(signal, trades, prices, horizon=21, prior_strength=20.0):\n    """A score_fn(as_of) for the backtest engine that re-estimates member skill\n    walk-forward at each date and feeds it to the congressional signal as member\n    weights. Forward returns are precomputed once for speed.\n    """\n    buy_rets = precompute_buy_returns(trades, prices, horizon=horizon)\n\n    def score_fn(as_of):\n        mult = member_skill(buy_rets, as_of, prior_strength=prior_strength)\n        r = signal.compute(as_of, trades=trades,\n                           member_weights=(mult if len(mult) else None))\n        return r["score"] if len(r) else None\n\n    return score_fn\n')
open("src/quant_research/ingestion/client.py", "w").write('"""A unified client for Quiver Quantitative data.\n\nThe QuiverClient exposes one method per dataset. Each method fetches raw data\n(from the live API when a token is supplied, otherwise from the mock generator)\nand returns it normalized into a consistent internal schema. The rest of the\nproject depends only on that internal schema, so nothing downstream needs to\nknow or care whether the data came from the live API or the mock source.\n"""\nimport re\nimport pandas as pd\n\nfrom . import mock_data\n\n\ndef _parse_range(text):\n    """Turn a Quiver amount range like \'$15,001 - $50,000\' into (low, high)."""\n    nums = re.findall(r"[\\d,]+", str(text))\n    vals = [int(n.replace(",", "")) for n in nums if n.replace(",", "").isdigit()]\n    if len(vals) >= 2:\n        return vals[0], vals[1]\n    if len(vals) == 1:\n        return vals[0], vals[0]\n    return 0, 0\n\n\nclass QuiverClient:\n    def __init__(self, token=None, mock=False, mock_history_days=40):\n        self.mock_history_days = mock_history_days\n        # No token means we cannot reach the live API, so we fall back to mock.\n        self.mock = mock or token is None\n        self._api = None\n        if not self.mock:\n            import quiverquant            # imported lazily; unused in mock mode\n            self._api = quiverquant.quiver(token)\n\n    def _fetch_live(self, fetch, label, retries=3, backoff=2.0):\n        """Call a live quiverquant endpoint defensively.\n\n        The library builds its frame with ``pd.DataFrame(json.loads(response))``, so a\n        non-tabular reply — most often a rate-limit or auth string — surfaces as an opaque\n        "DataFrame constructor not properly called!". This retries transient failures and\n        then raises a clear, actionable error instead.\n        """\n        import time\n        last = None\n        for attempt in range(retries):\n            try:\n                df = fetch()\n                if isinstance(df, pd.DataFrame) and len(df):\n                    return df\n                last = ValueError("empty response")\n            except Exception as e:               # includes the library\'s DataFrame error\n                last = e\n            if attempt < retries - 1:\n                time.sleep(backoff * (attempt + 1))\n        raise RuntimeError(\n            f"live {label} fetch failed after {retries} tries ({type(last).__name__}: {last}). "\n            "A non-tabular reply usually means the endpoint is rate-limited — wait a minute "\n            "and retry.")\n\n    # ---- Congressional trades -------------------------------------------\n    def congress_trades(self, historical=False):\n        if self.mock:\n            raw = mock_data.mock_congress_trading(\n                history_days=self.mock_history_days,\n                n=max(180, self.mock_history_days * 4))\n        else:\n            try:\n                raw = self._fetch_live(\n                    lambda: self._api.congress_trading(recent=not historical), "congress")\n            except RuntimeError:\n                if not historical:\n                    raise\n                import warnings\n                warnings.warn("bulk congress endpoint unavailable; using the recent endpoint, "\n                              "which returns less history.")\n                raw = self._fetch_live(\n                    lambda: self._api.congress_trading(recent=True), "congress")\n        return self._normalize_congress(raw)\n\n    @staticmethod\n    def _col(df, *names, default=""):\n        """First present column among `names`, else a default-filled Series.\n\n        The recent and bulk Quiver endpoints differ in their column names, so\n        every field is looked up tolerantly rather than assumed.\n        """\n        for n in names:\n            if n in df.columns:\n                return df[n]\n        return pd.Series([default] * len(df), index=df.index)\n\n    def _normalize_congress(self, df):\n        df = df.copy()\n        if "Range" in df.columns:\n            lows, highs = zip(*df["Range"].map(_parse_range)) if len(df) else ([], [])\n            amount_min, amount_max = list(lows), list(highs)\n        else:\n            amt = (self._col(df, "Amount", "Trade_Size_USD", "amount", default=0)\n                   .astype(str).str.replace(r"[$,]", "", regex=True))\n            amt = pd.to_numeric(amt, errors="coerce").fillna(0)\n            amount_min, amount_max = amt.tolist(), amt.tolist()\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "transaction_date": pd.to_datetime(self._col(df, "TransactionDate", "Traded"),\n                                               errors="coerce"),\n            "report_date": pd.to_datetime(self._col(df, "ReportDate", "Filed"),\n                                          errors="coerce"),\n            "representative": self._col(df, "Representative", "Name"),\n            "party": self._col(df, "Party"),\n            "chamber": self._col(df, "House", "Chamber"),\n            "transaction_type": self._col(df, "Transaction").astype(str).str.strip().str.title(),\n            "amount_min": amount_min,\n            "amount_max": amount_max,\n        })\n        return out\n\n    # ---- Insider transactions -------------------------------------------\n    def insider_trades(self):\n        raw = (mock_data.mock_insiders() if self.mock\n               else self._api.insiders())\n        return self._normalize_insiders(raw)\n\n    def _normalize_insiders(self, df):\n        df = df.copy()\n        code_map = {"P": "Purchase", "S": "Sale"}\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["Date"]),\n            "insider_name": df["Name"],\n            "title": df["Title"],\n            "transaction_type": df["TransactionCode"].map(code_map).fillna("Other"),\n            "shares": df["Shares"].astype(int),\n            "price_per_share": df["PricePerShare"].astype(float),\n        })\n        out["value"] = (out["shares"] * out["price_per_share"]).round(2)\n        return out\n\n    # ---- Government contracts --------------------------------------------\n    def gov_contracts(self):\n        if self.mock:\n            days = max(self.mock_history_days, 120)\n            raw = mock_data.mock_gov_contracts(history_days=days, n=max(80, days * 2))\n        else:\n            raw = self._fetch_live(self._api.gov_contracts, "gov_contracts")\n        return self._normalize_gov(raw)\n\n    def _normalize_gov(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "award_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "agency": df["Agency"],\n        })\n        return out\n\n    # ---- Lobbying --------------------------------------------------------\n    def lobbying(self):\n        if self.mock:\n            days = max(self.mock_history_days, 360)\n            raw = mock_data.mock_lobbying(history_days=days, n=max(140, days))\n        else:\n            raw = self._fetch_live(self._api.lobbying, "lobbying")\n        return self._normalize_lobbying(raw)\n\n    def _normalize_lobbying(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "filing_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "client": df["Client"],\n            "issue": df["Issue"],\n        })\n        return out\n\n    # ---- Off-exchange (dark pool) volume ---------------------------------\n    def off_exchange(self):\n        if self.mock:\n            days = max(self.mock_history_days, 90)\n            raw = mock_data.mock_offexchange(history_days=days)\n        else:\n            raw = self._fetch_live(self._api.offexchange, "off_exchange")\n        return self._normalize_offexchange(raw)\n\n    def _normalize_offexchange(self, df):\n        df = df.copy()\n        oe_vol = pd.to_numeric(self._col(df, "OTC_Total", "OffExchange", default=0), errors="coerce")\n        oe_short = pd.to_numeric(self._col(df, "OTC_Short", "Short", default=0), errors="coerce")\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "date": pd.to_datetime(self._col(df, "Date"), errors="coerce"),\n            "oe_volume": oe_vol,\n            "oe_short": oe_short,\n        })\n        if "DPI" in df.columns:\n            out["dpi"] = pd.to_numeric(df["DPI"], errors="coerce")\n        else:                                            # DPI = off-exchange short ratio\n            out["dpi"] = (out.oe_short / out.oe_volume.replace(0, pd.NA)).clip(0, 1)\n        return out\n')
open("src/quant_research/__init__.py", "w").write('"""Quant equity research from alternative-data signals."""\n__version__ = "0.2.0"\n')
print("wrote engine.py, congress.py, member_skill.py, client.py, __init__.py (v0.2.0)")

wrote engine.py, congress.py, member_skill.py, client.py, __init__.py (v0.2.0)


In [4]:
!pip install -q -e .

import importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## Backtest setup — a score-ranked universe

The same `2022`-start weekly schedule as `06`, with one change: the priced universe is the
top names **by composite score** rather than an alphabetical slice. `PRICE_N` caps the count
to keep the live price pull manageable on a tablet; raise it for a fuller test.


In [5]:
import pandas as pd, numpy as np
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal
from quant_research.signals.govcontracts import GovContractsSignal
from quant_research.signals.lobbying import LobbyingSignal
from quant_research.signals.composite import CompositeSignal
from quant_research.signals.member_skill import dynamic_congress_score_fn, precompute_buy_returns, member_skill
from quant_research.backtest.engine import (event_study, long_top_curve, long_short_curve,
                                            benchmark_curve, metrics, beta_alpha)

START = pd.Timestamp("2022-01-01")
HORIZON = 21
PRICE_N = 500          # priced universe size (top-N by composite score)

client = (QuiverClient(token=QUIVER_TOKEN, mock_history_days=600) if USE_LIVE
          else QuiverClient(mock=True, mock_history_days=1400))
congress = CongressSignal(client)
event_signals = {"congress": congress,
                 "gov_contracts": GovContractsSignal(client),
                 "lobbying": LobbyingSignal(client)}
composite = CompositeSignal(event_signals)
prefetch = {"congress": client.congress_trades(historical=True),
            "gov_contracts": client.gov_contracts(),
            "lobbying": client.lobbying()}

ranked = composite.compute(prefetch=prefetch)          # sorted by score, descending
universe = ranked.head(PRICE_N).index.tolist()         # top-N by score, not alphabetical
score_fn = composite.score_fn(prefetch=prefetch)

if USE_LIVE:
    from quant_research.backtest.prices import price_history
    prices = price_history(universe, START, pd.Timestamp.today().normalize())
else:
    KW = {"congress": "trades", "gov_contracts": "contracts", "lobbying": "filings"}
    rk = {nm: s.compute(pd.Timestamp.today(), **{KW[nm]: prefetch[nm]})["score"].rank(pct=True)
          for nm, s in event_signals.items()}
    days = pd.bdate_range(START, pd.Timestamp.today()); rng = np.random.default_rng(7)
    prices = pd.DataFrame(
        {t: 100*np.exp(np.cumsum(rng.normal(sum(0.0012*(rk[nm].get(t,0.5)-0.5) for nm in rk),
                                            0.010, len(days)))) for t in universe}, index=days)

rebal = pd.bdate_range(prices.index[5], prices.index[-1], freq="W-FRI")
print(f"{len(ranked)} ranked | priced top-{len(universe)} by score | {prices.shape[1]} priced | "
      f"{len(rebal)} weekly rebalances | {prices.index[0].date()} to {prices.index[-1].date()}")

https://api.quiverquant.com/beta/bulk/congresstrading
1364 ranked | priced top-500 by score | 482 priced | 233 weekly rebalances | 2022-01-03 to 2026-07-01


In [6]:
# fix: demean member excess against only the buys matured as-of each date (no month-cohort look-ahead)
import pathlib, importlib
p = pathlib.Path("src/quant_research/signals/member_skill.py")
s = p.read_text()
s = s.replace(
    '        out["period"] = out.report_date.dt.to_period("M")\n'
    '        out["excess"] = out.fwd_ret - out.groupby("period").fwd_ret.transform("mean")\n',
    '        out["period"] = out.report_date.dt.to_period("M")\n')
s = s.replace(
    '    matured = buy_rets[buy_rets.exit_date <= pd.Timestamp(as_of)]\n'
    '    if matured.empty:\n'
    '        return pd.Series(dtype=float)\n'
    '    grp = matured.groupby("representative").excess',
    '    matured = buy_rets[buy_rets.exit_date <= pd.Timestamp(as_of)].copy()\n'
    '    if matured.empty:\n'
    '        return pd.Series(dtype=float)\n'
    '    matured["excess"] = matured.fwd_ret - matured.groupby("period").fwd_ret.transform("mean")\n'
    '    grp = matured.groupby("representative").excess')
p.write_text(s)

import quant_research.signals.member_skill as msk
importlib.reload(msk)
from quant_research.signals.member_skill import dynamic_congress_score_fn, precompute_buy_returns, member_skill
print("patched member_skill.py — re-run section D")

patched member_skill.py — re-run section D


## A — The score-ranked universe

Measuring the composite on the top names by score, rather than an arbitrary alphabetical
slice, makes the IC reflect the part of the ranking a portfolio would actually act on. On
live data the difference is real: the alphabetical cap in `06` could miss whole sectors;
this prices the highest-conviction names first.


In [7]:
s = event_study(score_fn, prices, rebal, horizon=HORIZON)["summary"]
print(f"Composite IC on the score-ranked universe ({HORIZON}d): "
      f"{s['mean_ic']:+.3f} | positive {s['ic_positive_share']:.0%} of {s['n_periods']} periods")

Composite IC on the score-ranked universe (21d): +0.013 | positive 58% of 229 periods


## B — Long-only conviction curve and the beta/alpha split

The vendor trackers are long-only, so this builds the same thing: equal-weight the top
quintile by score and hold. The market-neutral long-short and the equal-weight benchmark are
shown alongside. The decomposition regresses the long-only return on the benchmark,

$$ r_{\text{strat}} = \alpha + \beta\, r_{\text{bench}} + \varepsilon $$

so a headline CAGR splits into the part explained by market exposure ($\beta$) and the
residual selection edge ($\alpha$). On live data the long-only beta sits near one and most
of the CAGR lands in the beta term — which is the honest answer to where published
double-digit returns come from.


In [8]:
lt_eq, lt_r = long_top_curve(score_fn, prices, rebal, horizon=HORIZON)
ls_eq, ls_r = long_short_curve(score_fn, prices, rebal, horizon=HORIZON)
bm_eq, bm_r = benchmark_curve(prices, rebal, horizon=HORIZON)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_scatter(y=lt_eq.values, name="long-only top quintile")
fig.add_scatter(y=ls_eq.values, name="long-short (market-neutral)")
fig.add_scatter(y=bm_eq.values, name="equal-weight benchmark")
fig.update_layout(height=430, title="Long-only vs market-neutral vs benchmark",
                  xaxis_title="non-overlapping holding period", yaxis_title="growth of $1")
fig.show()

mt = metrics(lt_r, HORIZON); ba = beta_alpha(lt_r, bm_r, horizon=HORIZON)
print(f"Long-only top quintile: CAGR {mt['cagr']*100:+.1f}% | Sharpe {mt['sharpe']:.2f} | "
      f"{mt['n_trades']} trades")
print(f"Decomposition vs benchmark: beta {ba['beta']:.2f} | annualized alpha "
      f"{ba['alpha_annual']*100:+.1f}% | R^2 {ba['r_squared']:.2f}")

Long-only top quintile: CAGR +11.1% | Sharpe 0.62 | 46 trades
Decomposition vs benchmark: beta 0.90 | annualized alpha -0.4% | R^2 0.82


## C — Static high-conviction congress

A second congressional signal with the feature blend tilted toward size relative to net
worth, committee alignment, and cluster — the attributes a backtest has reason to trust most
— against the base signal on the same dates.


In [9]:
base_c = CongressSignal(client)
hc_c   = CongressSignal(client, high_conviction=True)

def congress_fn(sig):
    return lambda d: (lambda r: r["score"] if len(r) else None)(
        sig.compute(d, trades=prefetch["congress"]))

rows = []
for label, sig in [("base congress", base_c), ("high-conviction", hc_c)]:
    st = event_study(congress_fn(sig), prices, rebal, horizon=HORIZON)["summary"]
    rows.append({"signal": label, "mean_ic": round(st["mean_ic"], 3),
                 "ic_pos_%": round(st["ic_positive_share"]*100, 0),
                 "ls_%/mo": round(st["mean_long_short"]*100, 3)})
print("Static tilt, same dates:")
pd.DataFrame(rows).set_index("signal")

Static tilt, same dates:


,mean_ic,ic_pos_%,ls_%/mo
signal,,,
base congress,0.014,57.0,0.260
high-conviction,0.014,57.0,0.238


## D — Dynamic member-skill weighting (walk-forward)

This estimates each member's realized edge from buys whose holding window has already
closed, shrinks it toward zero so thin records carry little weight, and re-fits at every
rebalance date — no future information enters. The estimate feeds the congressional signal as
a per-member multiplier. Early in the sample few members have matured trades, so the signal
starts close to the base and tilts as records accumulate.


In [10]:
dyn_fn = dynamic_congress_score_fn(base_c, prefetch["congress"], prices,
                                   horizon=HORIZON, prior_strength=20.0)

rows = []
for label, fn in [("base congress", congress_fn(base_c)),
                  ("high-conviction", congress_fn(hc_c)),
                  ("dynamic member-skill", dyn_fn)]:
    st = event_study(fn, prices, rebal, horizon=HORIZON)["summary"]
    rows.append({"signal": label, "mean_ic": round(st["mean_ic"], 3),
                 "ic_pos_%": round(st["ic_positive_share"]*100, 0),
                 "ls_%/mo": round(st["mean_long_short"]*100, 3)})
print(f"Congress variants at the {HORIZON}-day horizon:")
display(pd.DataFrame(rows).set_index("signal"))

# a look at the multipliers as of the last date
br = precompute_buy_returns(prefetch["congress"], prices, horizon=HORIZON)
mult = member_skill(br, prices.index[-1], prior_strength=20.0).sort_values(ascending=False)
print(f"\n{len(mult)} members scored from {len(br)} matured buys | "
      f"multiplier range [{mult.min():.2f}, {mult.max():.2f}]")
print("Most and least up-weighted members:")
display(pd.concat([mult.head(3), mult.tail(3)]).round(2).to_frame("multiplier"))

Congress variants at the 21-day horizon:


,mean_ic,ic_pos_%,ls_%/mo
signal,,,
base congress,0.014,57.0,0.260
high-conviction,0.014,57.0,0.238
dynamic member-skill,0.017,59.0,0.369



256 members scored from 24402 matured buys | multiplier range [0.37, 2.72]
Most and least up-weighted members:


,multiplier
representative,
A. Mitchell Jr. McConnell,2.72
Bill Flores,2.72
Dan Sullivan,2.72
Marie Newman,0.37
Roger W. Marshall,0.37
Tom Malinowski,0.37


## Reading the optimizations

- **Score-ranked universe** removes an arbitrary sampling choice; treat its IC as the more
  trustworthy estimate going forward.
- **Beta/alpha split** is the reconciliation with the vendor numbers made concrete: a
  long-only beta near one with most of the return in the beta term confirms the headline
  CAGR is largely market exposure, with a thin alpha residual.
- **Static tilt and dynamic weighting** are improvements to attempt, not guaranteed wins.
  Whether either lifts the IC above base is exactly what the tables answer; adopt a variant
  only when it clears base on this honest, walk-forward measurement. The dynamic estimate is
  bounded by the same disclosure lag and short member records discussed in `06`, so it
  sharpens a modest signal rather than transforming it.

Carry the best-performing congress variant and the score-ranked universe into the dashboard,
and present the long-only curve with its beta/alpha split so the headline number is shown
honestly rather than implied.


## Commit

In [11]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "v0.2: beta/alpha split, score-ranked universe, high-conviction and dynamic congress, robust live client"
!git push

[main 2bcff4f] v0.2: beta/alpha split, score-ranked universe, high-conviction and dynamic congress, robust live client
 5 files changed, 198 insertions(+), 16 deletions(-)
 create mode 100644 src/quant_research/signals/member_skill.py
Enumerating objects: 22, done.
Counting objects: 100% (22/22), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (12/12), 5.08 KiB | 2.54 MiB/s, done.
Total 12 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 7 local objects.
To https://github.com/balla-a12/quant-equity-research.git
   7b445ad..2bcff4f  main -> main


## Recap and next

The package is at `v0.2`: the backtest can split beta from alpha, the universe is ranked by
conviction, and the congressional signal can be tilted statically or weighted dynamically by
member skill. The final piece is the **dashboard** (`08`): today's top composite candidates
with their per-signal breakdown, the trending names per dataset, and this backtest evidence —
including the honest beta/alpha framing — attached so the whole system reads at a glance.
